# 02 — Feature Engineering

**Purpose:** Build the training dataset that XGBoost will learn from.

Raw data from `product_daily_features` isn't ready for training yet. We need to:
1. Filter out noise (stockout rows, missing lags)
2. Encode the product identifier as a number (ML models need numbers, not strings)
3. Construct multi-horizon targets — one row per (date, product, horizon)
4. Split into train and test sets

**Run after notebook 01.** Assumes `df` is already loaded — re-run Cell 1 from notebook 01 if starting fresh.

---

## Cell 1 — Re-load data (if starting fresh)

Skip if `df` is already in memory from notebook 01.

In [ ]:
import os
from pathlib import Path
from urllib.parse import quote_plus

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv(Path.cwd().parent.parent / ".env")
password = quote_plus(os.getenv("password", ""))
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('user')}:{password}"
    f"@{os.getenv('host')}:{os.getenv('port')}/{os.getenv('dbname')}",
    future=True, pool_pre_ping=True,
)

df = pd.read_sql(
    text("""
        SELECT date, product_id, quantity_sold,
               lag_1_qty, lag_7_qty,
               last_7_day_avg, last_30_day_avg, last_60_day_avg,
               last_7_day_stddev, day_of_week,
               is_holiday, days_to_next_festival, days_since_last_festival,
               stockout_proxy
        FROM derived.product_daily_features
        ORDER BY product_id, date
    """),
    con=engine, parse_dates=["date"],
)
print(f"Loaded {len(df):,} rows")

## Cell 2 — Filter: remove stockout rows and missing lags

We remove:
- `stockout_proxy = True` — biased zeros (shelf was empty, not zero demand)
- Rows where `lag_1_qty` or `lag_7_qty` is NULL — these are the very first days of a product's history, not useful for training since the model always expects lag features at inference time

**How many rows survive?** You should lose ~1–2% to stockouts and a small fraction to missing lags.

In [ ]:
df_clean = df[
    (~df["stockout_proxy"]) &
    (df["lag_1_qty"].notna()) &
    (df["lag_7_qty"].notna())
].copy()

print(f"Before filter: {len(df):,}")
print(f"After filter:  {len(df_clean):,}  ({len(df_clean)/len(df)*100:.1f}% retained)")

## Cell 3 — Encode `product_id` as integers

XGBoost needs numbers, not strings. We create a **deterministic label encoding**: sort all product_ids alphabetically, assign integer 0, 1, 2... in that order.

**Why deterministic?** We save this encoding to disk (`ml/model/product_encoder.json`). When we run inference next week, we load the same encoding so the model sees the same integer for the same product. If we used a random encoding, the model would be confused.

**What happens to new products?** If a new barcode appears after training, it won't be in the encoder. `predict.py` detects this and excludes those products — they fall back to WMA via COALESCE.

In [ ]:
# Build deterministic encoder
all_products = sorted(df_clean["product_id"].unique().tolist())
product_to_id = {p: i for i, p in enumerate(all_products)}

df_clean = df_clean.copy()
df_clean["product_id_encoded"] = df_clean["product_id"].map(product_to_id)

print(f"Total unique products encoded: {len(product_to_id):,}")
print(f"Sample mapping:")
for p, i in list(product_to_id.items())[:5]:
    print(f"  {p!r:20s} → {i}")

## Cell 4 — Build multi-horizon training set

This is the key feature engineering step. For each row at date T, we create **7 training rows** — one for each prediction horizon.

**What this looks like:**
```
date=2025-01-10, product=ABC, lag_1=5, ..., horizon=1, target=qty_sold[2025-01-11]
date=2025-01-10, product=ABC, lag_1=5, ..., horizon=2, target=qty_sold[2025-01-12]
...
date=2025-01-10, product=ABC, lag_1=5, ..., horizon=7, target=qty_sold[2025-01-17]
```

The model learns: *"given today's features, if I'm predicting 7 days out (horizon=7), I should account for weekend effects, etc."*

**Implementation:** `groupby('product_id')['quantity_sold'].shift(-h)` shifts each product's qty column backward by h positions, aligning tomorrow's (or day+h's) qty with today's feature row. Rows where the target is NULL (end of time series — no future data) are dropped.

In [ ]:
HORIZONS = list(range(1, 8))
FEATURE_COLS = [
    "lag_1_qty", "lag_7_qty",
    "last_7_day_avg", "last_30_day_avg", "last_60_day_avg",
    "last_7_day_stddev", "day_of_week",
    "is_holiday", "days_to_next_festival", "days_since_last_festival",
    "product_id_encoded", "horizon_days",
]

df_sorted = df_clean.sort_values(["product_id", "date"])
chunks = []
for h in HORIZONS:
    chunk = df_sorted.copy()
    chunk["target"] = df_sorted.groupby("product_id")["quantity_sold"].shift(-h)
    chunk["horizon_days"] = h
    chunks.append(chunk)

training_set = pd.concat(chunks, ignore_index=True).dropna(subset=["target"]).reset_index(drop=True)

print(f"Training set rows: {len(training_set):,}")
print(f"Rows per horizon: {len(training_set) / len(HORIZONS):,.0f} avg")
print(f"Horizon distribution:")
print(training_set["horizon_days"].value_counts().sort_index())

## Cell 5 — NULL audit on feature columns

Before training, check how many NULLs remain in each feature column. XGBoost handles NULLs natively (it learns a "missing value branch" for each split), but high null rates can weaken a feature's contribution.

**Expected:**
- `last_7_day_stddev`: some NULLs for products with < 7 days of history
- `is_holiday`, `days_to_next_festival`, `days_since_last_festival`: possibly 100% NULL if calendar_dim not seeded — that's fine, XGBoost will just ignore these features

In [ ]:
null_pct = training_set[FEATURE_COLS].isnull().mean() * 100
print("NULL % per feature column:")
print(null_pct.round(2).to_string())

## Cell 6 — Train/test split

We split by **time**, not randomly. This is critical for time series:
- **Train:** all rows where `date <= max_date - 30 days`
- **Test:** all rows where `date > max_date - 30 days`

**Why not random split?** If we randomly shuffle and split, some future dates end up in training and past dates in test. The model would effectively be peeking at the future. In production, we always predict forward — the test set must simulate this.

**Note:** The 'date' here is the **feature date** (the observation date), not the target date. A test row with feature_date=yesterday and horizon=7 has a target 7 days in the future — that's fine for evaluation purposes.

In [ ]:
TEST_DAYS = 30
cutoff = training_set["date"].max() - pd.Timedelta(days=TEST_DAYS)

train_set = training_set[training_set["date"] <= cutoff]
test_set  = training_set[training_set["date"] > cutoff]

X_train = train_set[FEATURE_COLS].fillna(0)
y_train = train_set["target"]
X_test  = test_set[FEATURE_COLS].fillna(0)
y_test  = test_set["target"]

print(f"Train rows: {len(X_train):,}  |  date range: {train_set['date'].min()} → {train_set['date'].max()}")
print(f"Test rows:  {len(X_test):,}  |  date range: {test_set['date'].min()} → {test_set['date'].max()}")
print(f"\nTarget stats (train):  mean={y_train.mean():.2f}  std={y_train.std():.2f}  max={y_train.max():.0f}")
print(f"Target stats (test):   mean={y_test.mean():.2f}  std={y_test.std():.2f}  max={y_test.max():.0f}")